<a href="https://colab.research.google.com/github/aziza-radouani/25007595-aziza-radouani-FIN3/blob/main/notebooks/projet_F3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# F3 — Prévision de l'inflation

**Nom et prénom :** Aziza Radouani    **Code Apogée :** 25007595    **Groupe :** FIN3    **Filière :** S7 FIN    **Date :** 17/11/2026

**Dépôt GitHub :** https://github.com/aziza-radouani/25007595-aziza-radouani-FIN3

## 1. Sujet et problématique

- **Intérêt du sujet** : L'inflation influence les décisions des banques centrales (taux directeurs), des entreprises (prix, salaires, contrats) et des investisseurs (obligations, pouvoir d'achat). Savoir si des variables simples comme le chômage et le taux directeur aident à anticiper l'inflation à un an permet d'éclairer ces décisions.
- **Problématique** : Le chômage et le taux directeur permettent-ils de prévoir l'inflation annuelle 12 mois plus tard mieux que la dernière valeur observée ?
- **Objectif mesurable** : Obtenir, sur la période de test (janvier 2015 jusqu'à la fin des données), un RMSE inférieur d'au moins 10 % à celui de la référence « dernière valeur observée » (l'inflation dans 12 mois est supposée égale à l'inflation d'aujourd'hui).
- **Nature du problème** : Régression sur série temporelle, car la cible (l'inflation annuelle à t+12) est numérique et les observations sont ordonnées dans le temps.
- **Algorithme principal envisagé** : n° 2, Ridge (régression linéaire avec pénalisation L2).

## 2. Jeu de données
## 3. Exploration
## 4. Préparation
## 5. Algorithme
## 6. Protocole expérimental
## 7. Résultats
## 8. Analyse critique
## 9. Conclusion
## 10. Références et reproduction
## Déclaration des ressources et outils d'IA

In [9]:
!git clone https://github.com/aziza-radouani/25007595-aziza-radouani-FIN3.git
%cd 25007595-aziza-radouani-FIN3
!ls data/raw

Cloning into '25007595-aziza-radouani-FIN3'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 48 (delta 13), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 27.17 KiB | 13.58 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/25007595-aziza-radouani-FIN3/25007595-aziza-radouani-FIN3
CPIAUCSL.csv  FEDFUNDS.csv  UNRATE.csv


In [10]:
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from pathlib import Path

warnings.filterwarnings('ignore')
GRAINE = 42
RAW = Path('data/raw')

print('Python', sys.version.split()[0], '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scikit-learn', sklearn.__version__)

Python 3.13.18 | numpy 2.4.6 | pandas 2.2.3 | scikit-learn 1.6.1


In [11]:
series = ['CPIAUCSL', 'UNRATE', 'FEDFUNDS']
morceaux = []
for s in series:
    d = pd.read_csv(RAW / f'{s}.csv', index_col=0, parse_dates=True)
    morceaux.append(pd.to_numeric(d.iloc[:, 0], errors='coerce').rename(s))
df = pd.concat(morceaux, axis=1)

print('Dimensions :', df.shape)
print('Période :', df.index.min().date(), '->', df.index.max().date())
print('Fréquence :', pd.infer_freq(df.index))
print('Valeurs manquantes :'); print(df.isna().sum())
display(df.head()); display(df.tail())

Dimensions : (958, 3)
Période : 1947-01-01 -> 2026-10-01
Fréquence : MS
Valeurs manquantes :
CPIAUCSL     1
UNRATE      13
FEDFUNDS    90
dtype: int64


,CPIAUCSL,UNRATE,FEDFUNDS
observation_date,,,
1947-01-01,21.48,NaN,NaN
1947-02-01,21.62,NaN,NaN
1947-03-01,22.00,NaN,NaN
1947-04-01,22.00,NaN,NaN
1947-05-01,21.95,NaN,NaN


,CPIAUCSL,UNRATE,FEDFUNDS
observation_date,,,
2026-06-01,332.568,4.2,3.63
2026-07-01,332.813,4.1,3.63
2026-08-01,334.131,4.1,3.63
2026-09-01,336.298,4.2,3.75
2026-10-01,337.476,4.1,3.88


In [13]:
from IPython.display import Markdown

n = len(df)
debuts = {c: df[c].first_valid_index().strftime('%Y-%m') for c in df.columns}
pct = (df.isna().sum() / n * 100).round(1)

lacunes = []
for c in df.columns:
    apres = df.loc[df[c].first_valid_index():, c]
    for d in apres[apres.isna()].index:
        lacunes.append(c + ' (' + d.strftime('%Y-%m') + ')')
texte_lacunes = ', '.join(lacunes) if lacunes else 'aucune'

lignes = [
    "## 2. Jeu de données",
    "",
    "- **Source** : FRED (Federal Reserve Bank of St. Louis), trois séries mensuelles : CPIAUCSL, UNRATE et FEDFUNDS (https://fred.stlouisfed.org/series/CPIAUCSL, https://fred.stlouisfed.org/series/UNRATE, https://fred.stlouisfed.org/series/FEDFUNDS).",
    "- **Date d'extraction** : 17/11/2026",
    "- **Licence** : à confirmer sur la page de chaque série FRED (rubrique Copyright).",
    "- **Taille** : " + str(n) + " lignes × " + str(df.shape[1]) + " variables, période " + df.index.min().strftime('%Y-%m') + " à " + df.index.max().strftime('%Y-%m') + ", fréquence mensuelle.",
    "- **Unité d'observation** : un mois (une ligne = un mois).",
    "- **Variable cible** : l'inflation annuelle à t+12, construite à partir de CPIAUCSL (voir section 4).",
    "- **Adéquation** : la fiche du sujet propose le chômage et le taux directeur pour prévoir l'inflation ; les séries sont mensuelles et couvrent plusieurs décennies.",
    "- **Limites** : (1) les séries ne commencent pas à la même date (CPIAUCSL : " + debuts['CPIAUCSL'] + ", UNRATE : " + debuts['UNRATE'] + ", FEDFUNDS : " + debuts['FEDFUNDS'] + ") ; (2) valeurs manquantes après le début des séries : " + texte_lacunes + " ; (3) les séries sont révisées après publication (biais d'anticipation) ; (4) la relation entre chômage, taux directeur et inflation change selon les régimes (années 1970, 2008, 2021-2022).",
    "",
    "| Variable | Description | Type | Unité | Rôle | Manquants (%) |",
    "| --- | --- | --- | --- | --- | --- |",
    "| CPIAUCSL | Indice des prix à la consommation | numérique | indice | base de la cible | " + str(pct['CPIAUCSL']) + " |",
    "| UNRATE | Taux de chômage | numérique | % | explicative | " + str(pct['UNRATE']) + " |",
    "| FEDFUNDS | Taux directeur effectif de la Fed | numérique | % | explicative | " + str(pct['FEDFUNDS']) + " |",
]
texte = "\n".join(lignes)
print(texte)

## 2. Jeu de données

- **Source** : FRED (Federal Reserve Bank of St. Louis), trois séries mensuelles : CPIAUCSL, UNRATE et FEDFUNDS (https://fred.stlouisfed.org/series/CPIAUCSL, https://fred.stlouisfed.org/series/UNRATE, https://fred.stlouisfed.org/series/FEDFUNDS).
- **Date d'extraction** : 17/11/2026
- **Licence** : à confirmer sur la page de chaque série FRED (rubrique Copyright).
- **Taille** : 958 lignes × 3 variables, période 1947-01 à 2026-10, fréquence mensuelle.
- **Unité d'observation** : un mois (une ligne = un mois).
- **Variable cible** : l'inflation annuelle à t+12, construite à partir de CPIAUCSL (voir section 4).
- **Adéquation** : la fiche du sujet propose le chômage et le taux directeur pour prévoir l'inflation ; les séries sont mensuelles et couvrent plusieurs décennies.
- **Limites** : (1) les séries ne commencent pas à la même date (CPIAUCSL : 1947-01, UNRATE : 1948-01, FEDFUNDS : 1954-07) ; (2) valeurs manquantes après le début des séries : CPIAUCSL (2025-10), U

## 2. Jeu de données

- **Source** : FRED (Federal Reserve Bank of St. Louis), trois séries mensuelles : CPIAUCSL, UNRATE et FEDFUNDS (https://fred.stlouisfed.org/series/CPIAUCSL, https://fred.stlouisfed.org/series/UNRATE, https://fred.stlouisfed.org/series/FEDFUNDS).
- **Date d'extraction** : 17/11/2026
- **Licence** : à confirmer sur la page de chaque série FRED (rubrique Copyright).
- **Taille** : 958 lignes × 3 variables, période 1947-01 à 2026-10, fréquence mensuelle.
- **Unité d'observation** : un mois (une ligne = un mois).
- **Variable cible** : l'inflation annuelle à t+12, construite à partir de CPIAUCSL (voir section 4).
- **Adéquation** : la fiche du sujet propose le chômage et le taux directeur pour prévoir l'inflation ; les séries sont mensuelles et couvrent plusieurs décennies.
- **Limites** : (1) les séries ne commencent pas à la même date (CPIAUCSL : 1947-01, UNRATE : 1948-01, FEDFUNDS : 1954-07) ; (2) valeurs manquantes après le début des séries : CPIAUCSL (2025-10), UNRATE (2025-10) ; (3) les séries sont révisées après publication (biais d'anticipation) ; (4) la relation entre chômage, taux directeur et inflation change selon les régimes (années 1970, 2008, 2021-2022).

| Variable | Description | Type | Unité | Rôle | Manquants (%) |
| --- | --- | --- | --- | --- | --- |
| CPIAUCSL | Indice des prix à la consommation | numérique | indice | base de la cible | 0.1 |
| UNRATE | Taux de chômage | numérique | % | explicative | 1.4 |
| FEDFUNDS | Taux directeur effectif de la Fed | numérique | % | explicative | 9.4 |
